# D2 · GDAL/OGR command-line tools — convert, reproject, clip

**What GDAL/OGR is:** the library under almost every GIS program. **GDAL** handles rasters, **OGR** handles vectors
(they are one project today). You have used it all along without seeing it: Rasterio, Pyogrio, Fiona, QGIS and PostGIS rasters all call it.
Here you use its **command-line tools** directly — the fastest way to convert, reproject and clip whole files.

| Tool | Job | Python twin you know |
|---|---|---|
| `ogrinfo` / `gdalinfo` | describe a vector / raster file | `pyogrio.read_info` / `rasterio.open().profile` |
| `ogr2ogr` | convert, reproject, filter, clip vectors (also into PostGIS) | `read_file` + `to_crs` + `to_file` |
| `gdal_translate` | convert or cut rasters | `rasterio` window + write |
| `gdalwarp` | reproject and clip rasters | `rasterio.warp.reproject`, `rasterio.mask` |
| `gdaldem` | slope, hillshade, aspect | numpy `gradient` (A5 4.3) |
| `gdal_rasterize`, `gdal_grid`, `gdal_contour`, `gdal_viewshed` | vector → raster, interpolation, contours, visibility | `rasterio.features.rasterize` |

**Daily picture:** Python libraries are like cooking with separate knives and pans. GDAL tools are the food processor:
one command, whole file, done — and exactly the same command works on Windows, Mac, Linux and servers.

In Jupyter you can write `!ogr2ogr ...`. We use the helper `sh("...")`, which also hides progress bars.
**Where this fits in your plan:** the professional's everyday file toolbox. Learn the 5 core commands well.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))      # lets the notebook find the `geotrain` helper
warnings.filterwarnings("ignore")
from geotrain import write_riverton, DATA_DIR
if not (DATA_DIR / "riverton.gpkg").exists():   # build the Riverton data the first time
    write_riverton()
GPKG = DATA_DIR / "riverton.gpkg"
print("Data folder:", DATA_DIR)

In [ ]:
from geotrain.desktop import qrun, qhelp, qlist, sh, grass_alg, enable_grass, qgis_process_path, quote, minimal_project
OUT = DATA_DIR / "d_out"
OUT.mkdir(exist_ok=True)
V = lambda layer: f"{GPKG}|layername={layer}"       # how QGIS names a layer inside a GeoPackage
DEM = DATA_DIR / "dem.tif"
print("qgis_process:", qgis_process_path())

In [ ]:
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from rasterio.plot import show
G, D, O = quote(GPKG), quote(DEM), OUT        # quoted paths for shell commands

---
## Level 1 — Basics: describe files

**Goal of this level:** read what is inside a vector or raster file from the command line.

**Daily-life picture:** Reading the label on a box before opening it.

### Exercise 1.1 — Describe a vector layer

**Step 1 · Purpose.** What is the **purpose** of `ogrinfo -so file layer`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`ogrinfo -so {G} schools`. Without a layer name, it lists the layers.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
sh(f"ogrinfo -so {G} ____")

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Prints a summary of a layer (`-so` = summary only): geometry type, feature count, extent, CRS and fields.

**Real life:** Checking a file from a partner before loading it.

```python
sh(f"ogrinfo -so {G} schools")
```

</details>

### Exercise 1.2 — Describe a raster

**Step 1 · Purpose.** What is the **purpose** of `gdalinfo [-stats] file`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`gdalinfo -stats {D}`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
sh(f"gdalinfo ____ {D}")

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Prints size, pixel size, CRS, corner coordinates, bands, nodata; `-stats` adds min/max/mean.

**Real life:** Checking the resolution and nodata value of a downloaded elevation model.

```python
sh(f"gdalinfo -stats {D}")
```

</details>

### Exercise 1.3 — Describe a CRS

**Step 1 · Purpose.** What is the **purpose** of `gdalsrsinfo -o proj4 EPSG:32633`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`-o proj4` gives the short form; `-o wkt2` the full modern form.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
sh("gdalsrsinfo -o ____ EPSG:32633")

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Prints a CRS definition in several formats (WKT, PROJ string, EPSG).

**Real life:** Translating a CRS for software that wants a PROJ string instead of an EPSG code.

```python
sh("gdalsrsinfo -o proj4 EPSG:32633")
```

</details>

### Exercise 1.4 — Value at one location

**Step 1 · Purpose.** What is the **purpose** of `gdallocationinfo -valonly -geoloc file x y`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Town hall ≈ 391440 5820080. Expect ~46.7 (same as A5 2.1).

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
sh(f"gdallocationinfo -valonly ____ {D} 391440 5820080")

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Prints the pixel value at a map coordinate (`-geoloc` = coordinates in the raster's CRS).

**Real life:** Quick checks: 'what is the elevation at the town hall?'

```python
sh(f"gdallocationinfo -valonly -geoloc {D} 391440 5820080")
```

</details>

---
## Level 2 — Core tools: ogr2ogr and gdal_translate / gdalwarp

**Goal of this level:** convert, reproject and filter vectors; cut, convert and reproject rasters.

**Daily-life picture:** Changing a document's format, language and page size in one go.

### Exercise 2.1 — Convert and reproject a vector

**Step 1 · Purpose.** What is the **purpose** of `ogr2ogr -f GeoJSON -t_srs EPSG:4326 out.geojson in.gpkg layer`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`ogr2ogr -f GeoJSON -t_srs EPSG:4326 {O}/schools.geojson {G} schools`. For formats like GeoPackage, `-overwrite` allows re-running; a GeoJSON file must be deleted first.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
(O / "schools.geojson").unlink(missing_ok=True)      # GeoJSON cannot be overwritten in place
sh(f"ogr2ogr -f GeoJSON -t_srs ____ {O}/schools.geojson {G} schools")
print(gpd.read_file(O / "schools.geojson").crs)

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Copies a layer into another format (`-f`) and CRS (`-t_srs` = target CRS). Note the order: **output first, then input**.

**Real life:** Publishing schools as GeoJSON in lon/lat for a web map (A4 3.1 in one command).

```python
(O / "schools.geojson").unlink(missing_ok=True)      # GeoJSON cannot be overwritten in place
sh(f"ogr2ogr -f GeoJSON -t_srs EPSG:4326 {O}/schools.geojson {G} schools")
print(gpd.read_file(O / "schools.geojson").crs)
```

</details>

### Exercise 2.2 — Filter with -where and -select

**Step 1 · Purpose.** What is the **purpose** of `ogr2ogr -where "..." -select a,b`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`-where "severity = 'fatal'"` and `-select date,hour`. Mind the quotes: double outside, single inside.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
sh(f"""ogr2ogr -overwrite -f GPKG {O}/fatal.gpkg {G} accidents -where "____" -select date,hour""")
sh(f"ogrinfo -so {O}/fatal.gpkg accidents")

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** `-where` keeps only rows matching an SQL condition; `-select` keeps only the listed fields.

**Real life:** Extract only fatal accidents with only date and hour, for a journalist (A4 test A).

```python
sh(f"""ogr2ogr -overwrite -f GPKG {O}/fatal.gpkg {G} accidents -where "severity = 'fatal'" -select date,hour""")
sh(f"ogrinfo -so {O}/fatal.gpkg accidents")
```

</details>

### Exercise 2.3 — SQL with spatial functions inside ogr2ogr

**Step 1 · Purpose.** What is the **purpose** of `ogr2ogr -dialect SQLite -sql "SELECT ST_Buffer(geom, 600) ..."`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

In a GeoPackage the geometry column is called `geom`. `-sql "SELECT school, ST_Buffer(geom, 600) AS geom FROM schools"` and `-nln catchments` (new layer name).

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
sh(f"""ogr2ogr -overwrite -f GPKG {O}/catch.gpkg {G} -dialect SQLite -nln catchments -sql "SELECT school, ____(geom, 600) AS geom FROM schools" """)
gpd.read_file(O / "catch.gpkg").plot(alpha=.4); plt.show()

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Runs an SQL query (with SpatiaLite spatial functions) on the file while converting — a small database query without a database.

**Real life:** Make school catchments or join tables in one command inside a data pipeline.

```python
sh(f"""ogr2ogr -overwrite -f GPKG {O}/catch.gpkg {G} -dialect SQLite -nln catchments -sql "SELECT school, ST_Buffer(geom, 600) AS geom FROM schools" """)
gpd.read_file(O / "catch.gpkg").plot(alpha=.4); plt.show()
```

</details>

### Exercise 2.4 — Cut and convert a raster

**Step 1 · Purpose.** What is the **purpose** of `gdal_translate -projwin ulx uly lrx lry / -of COG`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Oldtown: `-projwin 392000 5822000 394000 5820000`. Then `-of COG` for the whole DEM.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
sh(f"gdal_translate -q -projwin 392000 5822000 ____ ____ {D} {O}/oldtown_dem.tif")
sh(f"gdal_translate -q -of ____ {D} {O}/dem_cog.tif")
print(rasterio.open(O / "oldtown_dem.tif").shape, rasterio.open(O / "dem_cog.tif").profile.get("tiled"))

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Copies a raster, optionally cutting a box (`-projwin` in map coordinates: upper-left x, y, lower-right x, y) or changing format (`-of`).

**Real life:** Cut your city out of a national DEM; make a **Cloud-Optimised GeoTIFF** (COG) that web apps can read in small pieces.

```python
sh(f"gdal_translate -q -projwin 392000 5822000 394000 5820000 {D} {O}/oldtown_dem.tif")
sh(f"gdal_translate -q -of COG {D} {O}/dem_cog.tif")
print(rasterio.open(O / "oldtown_dem.tif").shape, rasterio.open(O / "dem_cog.tif").profile.get("tiled"))
```

</details>

### Exercise 2.5 — Reproject a raster

**Step 1 · Purpose.** What is the **purpose** of `gdalwarp -t_srs EPSG:4326 -r bilinear in.tif out.tif`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`gdalwarp -q -overwrite -t_srs EPSG:4326 -r bilinear {D} {O}/dem_4326.tif`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
sh(f"gdalwarp -q -overwrite -t_srs ____ -r ____ {D} {O}/dem_4326.tif")
print(rasterio.open(O / "dem_4326.tif").crs, rasterio.open(O / "dem_4326.tif").res)

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Warps a raster into a new CRS; `-r` chooses the resampling method (bilinear for continuous values, near for classes).

**Real life:** Aligning a DEM with climate data in lon/lat (A5 3.5 in one line).

```python
sh(f"gdalwarp -q -overwrite -t_srs EPSG:4326 -r bilinear {D} {O}/dem_4326.tif")
print(rasterio.open(O / "dem_4326.tif").crs, rasterio.open(O / "dem_4326.tif").res)
```

</details>

---
## Level 3 — Combining: clip, terrain, rasterize, interpolate

**Goal of this level:** cut rasters with polygons, derive terrain, and move between vector and raster.

**Daily-life picture:** Cutting a photo along a drawn outline, adding shading, and turning a list of points into a heat map.

### Exercise 3.1 — Clip a raster with a polygon

**Step 1 · Purpose.** What is the **purpose** of `gdalwarp -cutline file -cl layer -cwhere "..." -crop_to_cutline`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`-cutline {G} -cl neighbourhoods -cwhere "name='Hillcrest'" -crop_to_cutline`. Mean elevation should be ≈ 85 m (A5 3.1).

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
sh(f"""gdalwarp -q -overwrite -cutline {G} -cl neighbourhoods -cwhere "name='____'" -crop_to_cutline {D} {O}/hillcrest.tif""")
sh(f"gdalinfo -stats {O}/hillcrest.tif", quiet=True)
print(rasterio.open(O / "hillcrest.tif").read(1, masked=True).mean())

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Keeps only the pixels inside a polygon read from a vector file, and crops the output to its box.

**Real life:** DEM of one district; satellite image of a nature reserve.

```python
sh(f"""gdalwarp -q -overwrite -cutline {G} -cl neighbourhoods -cwhere "name='Hillcrest'" -crop_to_cutline {D} {O}/hillcrest.tif""")
sh(f"gdalinfo -stats {O}/hillcrest.tif", quiet=True)
print(rasterio.open(O / "hillcrest.tif").read(1, masked=True).mean())
```

</details>

### Exercise 3.2 — Terrain from a DEM

**Step 1 · Purpose.** What is the **purpose** of `gdaldem slope / hillshade / aspect`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`gdaldem slope {D} {O}/slope.tif` and `gdaldem hillshade {D} {O}/hillshade.tif`. Add `-q` for quiet.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
sh(f"gdaldem ____ -q {D} {O}/slope.tif")
sh(f"gdaldem ____ -q {D} {O}/hillshade.tif")
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
show(rasterio.open(O / "hillshade.tif"), ax=axs[0], cmap="gray"); show(rasterio.open(O / "slope.tif"), ax=axs[1], cmap="magma")
plt.show()

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Computes terrain products from an elevation raster: slope (degrees), hillshade (shaded relief), aspect (direction a slope faces).

**Real life:** Hillshade makes maps readable; slope limits where you can build or cycle comfortably.

```python
sh(f"gdaldem slope -q {D} {O}/slope.tif")
sh(f"gdaldem hillshade -q {D} {O}/hillshade.tif")
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
show(rasterio.open(O / "hillshade.tif"), ax=axs[0], cmap="gray"); show(rasterio.open(O / "slope.tif"), ax=axs[1], cmap="magma")
plt.show()
```

</details>

### Exercise 3.3 — Vector to raster: a population grid

**Step 1 · Purpose.** What is the **purpose** of `gdal_rasterize -a field -add -tr 250 250`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`-a residents -add -init 0 -tr 250 250 -te 390000 5818000 396000 5824000 -ot Float32 {G} -l houses {O}/pop250.tif`. The total should equal all residents.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
sh(f"gdal_rasterize -q -a ____ -add -init 0 -tr 250 250 -te 390000 5818000 396000 5824000 -ot Float32 {G} -l houses {O}/pop250.tif")
pop = rasterio.open(O / "pop250.tif").read(1)
print(pop.shape, pop.sum(), pop.max())

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Burns vector features into raster cells; `-a` takes the value from a field, `-add` sums values that fall in the same cell.

**Real life:** Residents per 250 m cell (A5 test C) — the input for many raster models.

```python
sh(f"gdal_rasterize -q -a residents -add -init 0 -tr 250 250 -te 390000 5818000 396000 5824000 -ot Float32 {G} -l houses {O}/pop250.tif")
pop = rasterio.open(O / "pop250.tif").read(1)
print(pop.shape, pop.sum(), pop.max())
```

</details>

### Exercise 3.4 — Interpolate sensor values (IDW)

**Step 1 · Purpose.** What is the **purpose** of `gdal_grid -zfield pm25 -a invdist:power=2`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`-zfield pm25 -a invdist:power=2:smoothing=0 -txe 390000 396000 -tye 5824000 5818000 -outsize 120 120 -ot Float32 -l sensors`. Daily maths: power 2 means a sensor twice as far counts 4 times less.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
sh(f"gdal_grid -q -zfield ____ -a invdist:power=2:smoothing=0 -txe 390000 396000 -tye 5824000 5818000 -outsize 120 120 -ot Float32 -l sensors {G} {O}/pm25_idw.tif")
ax = plt.subplots(figsize=(5, 5))[1]
show(rasterio.open(O / "pm25_idw.tif"), ax=ax, cmap="YlOrRd")
gpd.read_file(GPKG, layer="roads").plot(ax=ax, color="black", linewidth=.6)
gpd.read_file(GPKG, layer="sensors").plot(ax=ax, color="blue"); plt.show()

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Estimates a continuous surface from scattered points; inverse distance weighting (IDW) gives nearer sensors more weight.

**Real life:** An air-pollution map for every street from 18 sensors.

```python
sh(f"gdal_grid -q -zfield pm25 -a invdist:power=2:smoothing=0 -txe 390000 396000 -tye 5824000 5818000 -outsize 120 120 -ot Float32 -l sensors {G} {O}/pm25_idw.tif")
ax = plt.subplots(figsize=(5, 5))[1]
show(rasterio.open(O / "pm25_idw.tif"), ax=ax, cmap="YlOrRd")
gpd.read_file(GPKG, layer="roads").plot(ax=ax, color="black", linewidth=.6)
gpd.read_file(GPKG, layer="sensors").plot(ax=ax, color="blue"); plt.show()
```

IDW is a simple model: it cannot know that pollution follows roads. Compare the map with the roads — the test asks you to judge it.

</details>

### Exercise 3.5 — Contour lines

**Step 1 · Purpose.** What is the **purpose** of `gdal_contour -a elev -i 5`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`gdal_contour -q -a elev -i 5 {D} {O}/contours.gpkg -f GPKG` (delete an old file first with `-overwrite` not supported → remove it in Python).

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
(O / "contours.gpkg").unlink(missing_ok=True)
sh(f"gdal_contour -q -a elev -i ____ {D} {O}/contours.gpkg -f GPKG")
ct = gpd.read_file(O / "contours.gpkg")
ct.plot(column="elev", cmap="terrain", legend=True, figsize=(5, 5)); plt.show()

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Draws lines of equal elevation from a DEM (`-i` = interval, `-a` = name of the elevation field).

**Real life:** Classic topographic maps; checking where the land rises above a flood level.

```python
(O / "contours.gpkg").unlink(missing_ok=True)
sh(f"gdal_contour -q -a elev -i 5 {D} {O}/contours.gpkg -f GPKG")
ct = gpd.read_file(O / "contours.gpkg")
ct.plot(column="elev", cmap="terrain", legend=True, figsize=(5, 5)); plt.show()
```

</details>

---
## Level 4 — Professional: pipelines, databases and models

**Goal of this level:** load databases, model visibility, and build repeatable conversion pipelines.

**Daily-life picture:** A factory line: each machine does one job, and the line runs the same way every day.

### Exercise 4.1 — Load layers straight into PostGIS

**Scenario.** Load the `schools` and `houses` layers into PostGIS with `ogr2ogr` (no Python reading), as `schools_ogr` and `houses_ogr`, with a spatial index. Needs the database from B0.

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

`ogr2ogr -f PostgreSQL PG:"host=localhost user=geo password=geo dbname=geotrain" file layer -nln new_name -overwrite`. ogr2ogr creates the GIST index by default. Build the PG string from the DSN.

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
from urllib.parse import urlparse
from geotrain.db import DSN, sql
u = urlparse(DSN)
PG = f'PG:"host={u.hostname} port={u.port or 5432} user={u.username} password={u.password} dbname={u.path[1:]}"'
for layer in ["schools", "houses"]:
    sh(f"ogr2ogr -f ____ {PG} {G} {layer} -nln {layer}_ogr -overwrite")
sql("SELECT f_table_name, type, srid FROM geometry_columns WHERE f_table_name LIKE '%_ogr'")

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Data engineering (ETL into a database)

ogr2ogr is the standard ETL tool for PostGIS: fast, streams big files, handles CRS and field types. Use it for bulk loads; use `to_postgis` when the data is already a GeoDataFrame.

```python
from urllib.parse import urlparse
from geotrain.db import DSN, sql
u = urlparse(DSN)
PG = f'PG:"host={u.hostname} port={u.port or 5432} user={u.username} password={u.password} dbname={u.path[1:]}"'
for layer in ["schools", "houses"]:
    sh(f"ogr2ogr -f PostgreSQL {PG} {G} {layer} -nln {layer}_ogr -overwrite")
sql("SELECT f_table_name, type, srid FROM geometry_columns WHERE f_table_name LIKE '%_ogr'")
```

</details>

### Exercise 4.2 — What can a new lookout tower see?

**Scenario.** Riverton plans a 30 m lookout tower on the hilltop (391 225, 5 822 975). What share of the town is visible from the top? Which neighbourhoods see it least?

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

`gdal_viewshed -ox 391225 -oy 5822975 -oz 30 {D} out.tif` → 255 = visible, 0 = not. Zonal share per neighbourhood with `rasterio.mask` (A5 3.4).

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
import numpy as np
from rasterio.mask import mask
sh(f"gdal_viewshed -q -ox 391225 -oy 5822975 -oz ____ {D} {O}/viewshed.tif")
vs = rasterio.open(O / "viewshed.tif")
print("visible share of town:", round((vs.read(1) == 255).mean(), 3))
nbh = gpd.read_file(GPKG, layer="neighbourhoods")
nbh["visible"] = [float((mask(vs, [g], crop=True)[0] == 255).mean()) for g in nbh.geometry]
nbh.sort_values("visible")[["name", "visible"]].round(2)

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Modelling (visibility)

A viewshed model uses only the terrain: in flat Riverton almost everything is visible. Its big limit: **buildings and trees are missing**. With a surface model (DSM, e.g. from LiDAR) instead of a bare-earth DEM, the result would be very different.

```python
import numpy as np
from rasterio.mask import mask
sh(f"gdal_viewshed -q -ox 391225 -oy 5822975 -oz 30 {D} {O}/viewshed.tif")
vs = rasterio.open(O / "viewshed.tif")
print("visible share of town:", round((vs.read(1) == 255).mean(), 3))
nbh = gpd.read_file(GPKG, layer="neighbourhoods")
nbh["visible"] = [float((mask(vs, [g], crop=True)[0] == 255).mean()) for g in nbh.geometry]
nbh.sort_values("visible")[["name", "visible"]].round(2)
```

</details>

### Exercise 4.3 — A repeatable conversion pipeline

**Scenario.** Write a function `publish(layer)` that exports a Riverton layer as (1) GeoJSON in EPSG:4326 and (2) FlatGeobuf in EPSG:32633, then returns the feature count of each output. Run it for all point layers.

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

Two `ogr2ogr` calls with `-f GeoJSON -t_srs EPSG:4326` and `-f FlatGeobuf`. Count with `pyogrio.read_info(path)["features"]`.

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
import pyogrio
def publish(layer):
    gj, fgb = O / f"{layer}.geojson", O / f"{layer}.fgb"
    gj.unlink(missing_ok=True); fgb.unlink(missing_ok=True)
    sh(f"ogr2ogr -f GeoJSON -t_srs EPSG:4326 {gj} {G} {layer}")
    sh(f"ogr2ogr -f ____ {fgb} {G} {layer}")
    return {"layer": layer, "geojson": pyogrio.read_info(gj)["features"], "fgb": pyogrio.read_info(fgb)["features"]}
import pandas as pd
pd.DataFrame([publish(l) for l in ["schools", "clinics", "sensors", "accidents"]])

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Data engineering (automation)

Small functions around GDAL commands make data publishing boring and reliable — which is exactly what you want. The same commands can run on a server every night.

```python
import pyogrio
def publish(layer):
    gj, fgb = O / f"{layer}.geojson", O / f"{layer}.fgb"
    gj.unlink(missing_ok=True); fgb.unlink(missing_ok=True)
    sh(f"ogr2ogr -f GeoJSON -t_srs EPSG:4326 {gj} {G} {layer}")
    sh(f"ogr2ogr -f FlatGeobuf {fgb} {G} {layer}")
    return {"layer": layer, "geojson": pyogrio.read_info(gj)["features"], "fgb": pyogrio.read_info(fgb)["features"]}
import pandas as pd
pd.DataFrame([publish(l) for l in ["schools", "clinics", "sensors", "accidents"]])
```

</details>

### GDAL in Python (optional)

GDAL also has its own Python API (`from osgeo import gdal, ogr`), with functions that mirror the tools:
`gdal.Warp(...)`, `gdal.Translate(...)`, `gdal.VectorTranslate(...)`, `gdal.DEMProcessing(...)`.
Install it with conda (`conda install -c conda-forge gdal`) or OSGeo4W. For most work, the command-line tools above plus Rasterio/Pyogrio are enough.

---
## 🏁 Final test

Use GDAL/OGR tools through `sh()`. For each: **question type → command(s) → check → interpretation.**

### Test 1 — 🛠️ Task question

**A.** Export the neighbourhoods as a **Shapefile in EPSG:4326** with only `name` and `population`. Check the result with `ogrinfo`.

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

```python
sh(f"ogr2ogr -overwrite -f 'ESRI Shapefile' -t_srs EPSG:4326 -select name,population {O}/nbh_4326.shp {G} neighbourhoods")
sh(f"ogrinfo -so {O}/nbh_4326.shp nbh_4326")
```

</details>

### Test 2 — 🛠️ Task question

**B.** Make a **100 m** resolution version of the DEM (from 50 m) with averaging, and compare the maximum elevation before and after.

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

```python
sh(f"gdalwarp -q -overwrite -tr 100 100 -r average {D} {O}/dem100.tif")
print(rasterio.open(DEM).read(1, masked=True).max(), rasterio.open(O / "dem100.tif").read(1, masked=True).max())
```
Averaging lowers the peak: coarser rasters smooth extremes (see A5 test E on resolution).

</details>

### Test 3 — 🛠️ Task question

**C.** Keep only the houses inside **Oldtown** using `ogr2ogr -clipsrc` and count them.

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

```python
sh(f"""ogr2ogr -overwrite -f GPKG {O}/oldtown_houses.gpkg {G} houses -clipsrc {G} -clipsrclayer neighbourhoods -clipsrcwhere "name='Oldtown'" """)
print(pyogrio.read_info(O / "oldtown_houses.gpkg")["features"])
```

</details>

### Test 4 — 🧠 Modelling question

**D · Modelling air quality.** The IDW map (3.4) ignores roads. Propose a better model for PM2.5 across Riverton, using tools from this course.

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

**Land-use regression:** explain PM2.5 at the sensors with predictors (distance to primary roads, road length within 300 m, greenness NDVI, population density), then predict on a grid.
Steps: predictor rasters with `gdal_proximity`/`gdal_rasterize`/NDVI (A5) → sample at sensors → regression (`spreg`, A6, and check the residuals' spatial autocorrelation) → apply the model to every cell.
Or **kriging** of the residuals (regression-kriging). Recommendation: land-use regression, because pollution is driven by roads, which IDW cannot see.

</details>

---
## 🏗️ Project — An open-data release with GDAL/OGR

**The brief.** Riverton publishes an **open-data package**. Using only GDAL/OGR command-line tools, produce: cloud-optimised rasters
        (elevation, hillshade, temperature), one GeoPackage with the main vector layers, a web GeoJSON, a district extract,
        and the same vectors in a PostGIS schema `opendata` for the city's map server. Then check everything.

**Your data** (made for this project, in `data/projects/`):

| Layer / file | What it holds |
|---|---|
| `dem.tif, lst.tif` | elevation and land-surface temperature rasters |
| `streets.gpkg, buildings.gpkg, tracts.gpkg` | vector layers |
| `riverton.gpkg → neighbourhoods` | for the district extract |

**How to work:** for each task write the plan in words, then the code, then one sentence with the result. Model answers are hidden; open them only after you tried. The last part happens in **QGIS**.

In [ ]:
from geotrain.projects import build_projects, PROJ_DIR, publish
build_projects()
P = lambda name, layer=None: f"{PROJ_DIR / name}" + (f"|layername={layer}" if layer else "")

import pandas as pd, pyogrio, rasterio
REL = OUT / "release"

### Project task 1

**Rasters as COG.** Write `dem_cog.tif`, `hillshade_cog.tif` (from `gdaldem`) and `lst_cog.tif` into `release/` as Cloud-Optimised GeoTIFFs.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
REL.mkdir(exist_ok=True)
sh(f"gdal_translate -q -of COG {quote(DEM)} {quote(REL / 'dem_cog.tif')}")
sh(f"gdaldem hillshade -q {quote(DEM)} {quote(OUT / 'hs_tmp.tif')}")
sh(f"gdal_translate -q -of COG {quote(OUT / 'hs_tmp.tif')} {quote(REL / 'hillshade_cog.tif')}")
sh(f"gdal_translate -q -of COG {quote(PROJ_DIR / 'lst.tif')} {quote(REL / 'lst_cog.tif')}")
print(sorted(p.name for p in REL.glob("*.tif")))
```

</details>

### Project task 2

**One GeoPackage, three layers.** Put streets, buildings and tracts into `release/riverton_open_data.gpkg` with `ogr2ogr` (first layer creates the file, the next ones use `-update`).

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
pkg = REL / "riverton_open_data.gpkg"
pkg.unlink(missing_ok=True)
for i, name in enumerate(["streets", "buildings", "tracts"]):
    extra = "" if i == 0 else "-update"
    sh(f"ogr2ogr {extra} -f GPKG -nln {name} {quote(pkg)} {quote(PROJ_DIR / (name + '.gpkg'))}")
sh(f"ogrinfo -so {quote(pkg)}")
```

</details>

### Project task 3

**For the web and for one district.** (a) tracts as GeoJSON in EPSG:4326 with only `tract_id`, `population`; (b) the buildings of **Oldtown** only (`-clipsrc` with the neighbourhood).

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
(REL / "tracts.geojson").unlink(missing_ok=True)
sh(f"ogr2ogr -f GeoJSON -t_srs EPSG:4326 -select tract_id,population {quote(REL / 'tracts.geojson')} {quote(PROJ_DIR / 'tracts.gpkg')}")
sh(f"""ogr2ogr -overwrite -f GPKG {quote(REL / 'oldtown_buildings.gpkg')} {quote(PROJ_DIR / 'buildings.gpkg')} -clipsrc {quote(GPKG)} -clipsrclayer neighbourhoods -clipsrcwhere "name='Oldtown'" """)
print(pyogrio.read_info(REL / "oldtown_buildings.gpkg")["features"], "buildings in Oldtown")
```

</details>

### Project task 4

**Into PostGIS** schema `opendata` with `ogr2ogr` (`-lco SCHEMA=opendata`), all three layers.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
from urllib.parse import urlparse
from geotrain.db import DSN, sql
u = urlparse(DSN)
PGC = f'PG:"host={u.hostname} port={u.port or 5432} user={u.username} password={u.password} dbname={u.path[1:]}"'
sql("CREATE SCHEMA IF NOT EXISTS opendata")
for name in ["streets", "buildings", "tracts"]:
    sh(f"ogr2ogr -f PostgreSQL {PGC} {quote(PROJ_DIR / (name + '.gpkg'))} -nln {name} -lco SCHEMA=opendata -overwrite")
sql("SELECT f_table_name, type, srid FROM geometry_columns WHERE f_table_schema = 'opendata' ORDER BY 1")
```

</details>

### Project task 5

**Release check.** A table with every file in `release/`: size (kB), and feature count or raster size — the 'contents' page of the package.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
rows = []
for f in sorted(REL.iterdir()):
    if f.suffix == ".tif":
        with rasterio.open(f) as r:
            rows.append({"file": f.name, "kB": f.stat().st_size // 1024, "content": f"raster {r.width}×{r.height}, {r.crs}"})
    elif f.suffix in (".gpkg", ".geojson"):
        rows.append({"file": f.name, "kB": f.stat().st_size // 1024,
                     "content": ", ".join(f"{n} ({pyogrio.read_info(f, layer=n)['features']})" for n, _ in pyogrio.list_layers(f))})
pd.DataFrame(rows)
```

</details>

### 🗺️ Project in QGIS

1. Drag `release/dem_cog.tif` and `hillshade_cog.tif` into QGIS; set the hillshade to *Multiply* blending over the DEM for a nice relief map.
2. Open `riverton_open_data.gpkg`: QGIS shows its three layers in one dialog.
3. Load **geotrain → opendata → buildings** from PostGIS: same data, now served by the database.
4. Check the COG: *Layer Properties → Information* shows overviews and internal tiling — that is what makes COGs fast on the web.

<details><summary>🔌 <b>How to connect QGIS to PostGIS</b> (once per computer)</summary>

1. Start the database (Docker: `docker start geotrain-db`).
2. In QGIS, open the **Browser** panel → right-click **PostgreSQL** → **New Connection…**
3. Name `geotrain`, Host `localhost`, Port `5432`, Database `geotrain`.
   Authentication → *Basic*: user `geo`, password `geo` (tick *Store*). Click **Test Connection** → OK.
4. Expand **geotrain → projects** (the schema) and **drag a table onto the map**.
5. To run SQL and see the result on the map: **Database → DB Manager → PostGIS → geotrain → SQL Window**.
   Write the query → **Execute** → tick **Load as new layer**, choose the geometry column and a unique id column → **Load**.
6. To open a GeoPackage instead: drag the `.gpkg` file from your file manager into QGIS.

</details>

### 📦 Deliverables (tick when done)

- [ ] `release/` folder: 3 COGs, 1 GeoPackage, 1 GeoJSON, 1 district extract
- [ ] Schema `opendata` in PostGIS
- [ ] Contents table of the package

### 🪞 Reflection — your long-term plan

Which file conversions do you do by hand today? Write the one `ogr2ogr` or `gdalwarp` command that would replace them.

✍️ **Your answer:** …